In [2]:
import pandas as pd
import os
from Bio import Entrez, SeqIO
import tqdm
from collections import Counter
import numpy as np
from scipy.stats import multinomial
import numpy as np
import ast


In [5]:
dict_cols = ['G>Aproportions', 'C>Tproportions', 'A>Gproportions', 'T>Cproportions']
iterator = pd.read_csv("/Users/reem/Mov/final_results_llrs_26.tsv",sep="\t", converters={col: ast.literal_eval for col in dict_cols},chunksize=1000)
df = pd.concat([chunk for chunk in tqdm.tqdm(iterator, desc='Loading data')])


Loading data: 16922it [05:05, 55.42it/s]


In [2]:
def generate_all_possible_contexts(sub):
    bases = ["A","C","G","T"]
    possible_contexts = []
    for a in bases:
        for b in bases:
            possible_contexts.append(f"{a}[{sub}]{b}")
    return possible_contexts
#print(generate_all_possible_contexts("G>A"))

In [ ]:
df["G>Aproportions"] = (
    df["G>Aproportions"].apply(lambda x: ast.literal_eval(x) if x not in ["", "nan", "Counter"] else {}))
print(df["G>Aproportions"].head())
print(type(df["G>Aproportions"].iloc[0]))

0    {}
1    {}
2    {}
3    {}
4    {}
Name: G>Aproportions, dtype: object
<class 'pandas.core.series.Series'>


In [14]:
# Compute mean context probabilities
def get_mean_context_probs(df, llr_condition, prop_col, llr_col='LLR'):
    """
    Filters dataframe into llr>6 and llr<6 and computes mean 
    proportions per substitution per context

    Returns a flattened dataframe with mutational contexts,seqname,LLR
    and computed means as the last row

    """

    df_filtered = df[llr_condition(df[llr_col])].copy()
    
    pivot = pd.json_normalize(df_filtered[prop_col]).fillna(0)
    pivot = pivot.loc[(pivot != 0).any(axis=1)]

    return pivot.mean()

print(get_mean_context_probs(df, llr_condition=lambda x: x > 6, prop_col="A>Gproportions", llr_col='LLR'))
def build_prob_table(df,prop_col,llr_high,llr_low,llr_col='LLR'):
    """
    Builds a table of mean context probabilities (Molnupiravir vs Normal)
    for a specific substitution type.
    """
    llr_high=6
    llr_low=6

    Mov_probs = get_mean_context_probs(df.loc[(df!=0).any(axis=1)], lambda x: x > 6, prop_col="A>Gproportions")
    Normal_probs  = get_mean_context_probs(df.loc[(df!=0).any(axis=1)], lambda x: x < 6, prop_col='A>Gproportions')
    all_contexts = generate_all_possible_contexts("A>G")

    # Match means to all possible contexts
    Mov_probs = Mov_probs.reindex(all_contexts, fill_value=0)
    Normal_probs = Normal_probs.reindex(all_contexts, fill_value=0)

    print(Mov_probs)
    print(Normal_probs)

    df_prob = pd.DataFrame({'Mutational_Context':all_contexts,
                           'Molnupiravir': Mov_probs.values,
                            'Normal': Normal_probs.values
                            }) 

    
    return df_prob


df_prob = build_prob_table(df,prop_col='A>Gproportions',llr_high=6, llr_low=6, llr_col='LLR')

print(df_prob)

A[A>G]A    0.133917
C[A>G]T    0.052255
T[A>G]A    0.046675
G[A>G]A    0.042780
C[A>G]A    0.062813
A[A>G]T    0.109217
C[A>G]C    0.038407
A[A>G]C    0.049782
G[A>G]G    0.039147
C[A>G]G    0.091455
T[A>G]T    0.044716
A[A>G]G    0.142126
G[A>G]C    0.016757
G[A>G]T    0.059952
T[A>G]G    0.050090
T[A>G]C    0.019911
dtype: float64
A[A>G]A    0.133917
A[A>G]C    0.049782
A[A>G]G    0.142126
A[A>G]T    0.109217
C[A>G]A    0.062813
C[A>G]C    0.038407
C[A>G]G    0.091455
C[A>G]T    0.052255
G[A>G]A    0.042780
G[A>G]C    0.016757
G[A>G]G    0.039147
G[A>G]T    0.059952
T[A>G]A    0.046675
T[A>G]C    0.019911
T[A>G]G    0.050090
T[A>G]T    0.044716
dtype: float64
A[A>G]A    0.067241
A[A>G]C    0.061043
A[A>G]G    0.119216
A[A>G]T    0.101340
C[A>G]A    0.046222
C[A>G]C    0.038284
C[A>G]G    0.068765
C[A>G]T    0.077134
G[A>G]A    0.036013
G[A>G]C    0.021386
G[A>G]G    0.030448
G[A>G]T    0.071873
T[A>G]A    0.033508
T[A>G]C    0.050322
T[A>G]G    0.071121
T[A>G]T    0.106084
dtype: flo

In [16]:
print(df_prob)
print(df_prob["Molnupiravir"].sum())
print(df_prob["Normal"].sum())

   Mutational_Context  Molnupiravir    Normal
0             A[A>G]A      0.133917  0.067241
1             A[A>G]C      0.049782  0.061043
2             A[A>G]G      0.142126  0.119216
3             A[A>G]T      0.109217  0.101340
4             C[A>G]A      0.062813  0.046222
5             C[A>G]C      0.038407  0.038284
6             C[A>G]G      0.091455  0.068765
7             C[A>G]T      0.052255  0.077134
8             G[A>G]A      0.042780  0.036013
9             G[A>G]C      0.016757  0.021386
10            G[A>G]G      0.039147  0.030448
11            G[A>G]T      0.059952  0.071873
12            T[A>G]A      0.046675  0.033508
13            T[A>G]C      0.019911  0.050322
14            T[A>G]G      0.050090  0.071121
15            T[A>G]T      0.044716  0.106084
0.9999999999999999
1.0


In [17]:
df_prob.to_csv("/Users/reem/A>Gprobs_new.tsv",sep="\t")

In [14]:
df["T>C_counts"] = (
    df["T>C_counts"]
    .astype(str)
    .str.replace("Counter(", "")  # remove "Counter("
    .str.rstrip(")")                           # remove trailing ")"
    .apply(lambda x: ast.literal_eval(x) if x not in ["", "nan", "Counter"] else {})
)
print(df["T>C_counts"].head())
print(type(df["T>C_counts"]))

0    {'A[T>C]A': 1}
1    {'A[T>C]A': 1}
2                {}
3    {'A[T>C]G': 1}
4                {}
Name: T>C_counts, dtype: object
<class 'pandas.core.series.Series'>


In [15]:
def calculate_llr(count_dict, pM, pN, contexts):
    counts = np.array([count_dict.get(ctx, 0) for ctx in contexts])
    n = counts.sum()
    if n == 0:
        return np.nan
    llM = multinomial.logpmf(counts, n=n, p=pM)
    llN = multinomial.logpmf(counts, n=n, p=pN)
    return float(llM - llN)

df_prob = pd.read_csv("/Users/reem/T>Cprobs_new.tsv",sep="\t")
contexts = df_prob["Mutational_Context"].values.tolist()
pM = df_prob["Molnupiravir"].values.tolist()
pN = df_prob["Normal"].values.tolist()
df[f"T>C_llr"] = df["T>C_counts"].apply(lambda x: calculate_llr(x, pM, pN, contexts))

print(df[f"T>C_llr"])


0          -0.661397
1          -0.661397
2                NaN
3          -0.356234
4                NaN
              ...   
16921560         NaN
16921561         NaN
16921562         NaN
16921563         NaN
16921564         NaN
Name: T>C_llr, Length: 16921565, dtype: float64


In [17]:
df.head()

,Unnamed: 0,seqName,privateNucMutations.unlabeledSubstitutions,subs,Counts,context,spectrum,G>A_counts,A>G_counts,C>T_counts,T>C_counts,G>Aproportions,A>Gproportions,C>Tproportions,T>Cproportions,LLR,G>A_llr,A>G_llr,C>T_llr,T>C_llr
0,0,hCoV-19/Indonesia/JK-GS-GSILab-878752/2022|202...,"A9364G,T9901C","A>G,T>C","{'A>G': 1, 'T>C': 1}","CAC,ATA","C[A>G]C,A[T>C]A",{},{'C[A>G]C': 1},{},{'A[T>C]A': 1},{},{'C[A>G]C': 1.0},{},{'A[T>C]A': 1.0},-0.739348,NaN,0.003207,NaN,-0.661397
1,1,hCoV-19/Germany/BW-RKI-I-1062047/2021|2021-10-...,"C455T,C1387T,G3338T,C6730T,C6936T,A13625G,T158...","C>T,C>T,G>T,C>T,C>T,A>G,T>G,G>T,G>A,G>T,A>C,C>...","{'C>T': 6, 'G>T': 4, 'A>G': 1, 'T>G': 1, 'G>A'...","ACT,ACA,AGT,ACA,TCT,GAT,CTT,CGT,AGT,CGC,AAA,AC...","A[C>T]T,A[C>T]A,A[G>T]T,A[C>T]A,T[C>T]T,G[A>G]...",{'A[G>A]T': 1},{'G[A>G]T': 1},"{'A[C>T]A': 2, 'A[C>T]T': 1, 'T[C>T]T': 1, 'A[...",{'A[T>C]A': 1},{'A[G>A]T': 1.0},{'G[A>G]T': 1.0},"{'A[C>T]T': 0.16666666666666666, 'A[C>T]A': 0....",{'A[T>C]A': 1.0},-6.493157,-0.337129,-0.181359,-0.373305,-0.661397
2,2,hCoV-19/South Korea/KDCA36211s/2022|2022-11-16...,NaN,NaN,{'': 1},NaN,NaN,{},{},{},{},{},{},{},{},0.000000,NaN,NaN,NaN,NaN
3,3,hCoV-19/Germany/MV-RKI-I-641916/2022|2022-02-2...,T16494C,T>C,{'T>C': 1},ATG,A[T>C]G,{},{},{},{'A[T>C]G': 1},{},{},{},{'A[T>C]G': 1.0},-0.528901,NaN,NaN,NaN,-0.356234
4,4,hCoV-19/Germany/BW-RKI-I-354764/2021|2021-11-2...,"C2227A,G4207T,C13673A,A21603G,C21658T,C23655T,...","C>A,G>T,C>A,A>G,C>T,C>T,C>T,G>T,C>T","{'C>A': 2, 'G>T': 2, 'A>G': 1, 'C>T': 4}","CCT,CGA,TCT,CAG,TCA,TCA,ACC,CGA,ACG","C[C>A]T,C[G>T]A,T[C>A]T,C[A>G]G,T[C>T]A,T[C>T]...",{},{'C[A>G]G': 1},"{'T[C>T]A': 2, 'A[C>T]C': 1, 'A[C>T]G': 1}",{},{},{'C[A>G]G': 1.0},"{'T[C>T]A': 0.5, 'A[C>T]C': 0.25, 'A[C>T]G': 0...",{},-4.058362,NaN,0.285154,-0.176959,NaN


In [25]:
df["sum_llrs"] = df["LLR"].fillna(0)+df[f"T>C_llr"].fillna(0)+df[f"C>T_llr"].fillna(0)+df[f"A>G_llr"].fillna(0)+df[f"G>A_llr"].fillna(0)
df["sum_llrs"]

0          -1.397538
1          -8.046346
2           0.000000
3          -0.885135
4          -3.950167
              ...   
16921560    0.521315
16921561    0.032029
16921562    2.053847
16921563    0.354069
16921564   -2.049079
Name: sum_llrs, Length: 16921565, dtype: float64

In [26]:
df.to_csv("/Users/reem/Mov/final_llrs26_per_context.tsv", sep="\t", index=False)

In [7]:
iterator = pd.read_csv("/Users/reem/Mov/final_llrs26_per_context.tsv",sep="\t",chunksize=1000)
df = pd.concat([chunk for chunk in tqdm.tqdm(iterator, desc='Loading data')])
df.head()

Loading data: 16922it [00:51, 328.23it/s]


,Unnamed: 0,seqName,privateNucMutations.unlabeledSubstitutions,subs,Counts,context,spectrum,G>A_counts,A>G_counts,C>T_counts,...,G>Aproportions,A>Gproportions,C>Tproportions,T>Cproportions,LLR,G>A_llr,A>G_llr,C>T_llr,T>C_llr,sum_llrs
0,0,hCoV-19/Indonesia/JK-GS-GSILab-878752/2022|202...,"A9364G,T9901C","A>G,T>C","{'A>G': 1, 'T>C': 1}","CAC,ATA","C[A>G]C,A[T>C]A",{},{'C[A>G]C': 1},{},...,{},{'C[A>G]C': 1.0},{},{'A[T>C]A': 1.0},-0.739348,NaN,0.003207,NaN,-0.661397,-1.397538
1,1,hCoV-19/Germany/BW-RKI-I-1062047/2021|2021-10-...,"C455T,C1387T,G3338T,C6730T,C6936T,A13625G,T158...","C>T,C>T,G>T,C>T,C>T,A>G,T>G,G>T,G>A,G>T,A>C,C>...","{'C>T': 6, 'G>T': 4, 'A>G': 1, 'T>G': 1, 'G>A'...","ACT,ACA,AGT,ACA,TCT,GAT,CTT,CGT,AGT,CGC,AAA,AC...","A[C>T]T,A[C>T]A,A[G>T]T,A[C>T]A,T[C>T]T,G[A>G]...",{'A[G>A]T': 1},{'G[A>G]T': 1},"{'A[C>T]A': 2, 'A[C>T]T': 1, 'T[C>T]T': 1, 'A[...",...,{'A[G>A]T': 1.0},{'G[A>G]T': 1.0},"{'A[C>T]T': 0.16666666666666666, 'A[C>T]A': 0....",{'A[T>C]A': 1.0},-6.493157,-0.337129,-0.181359,-0.373305,-0.661397,-8.046346
2,2,hCoV-19/South Korea/KDCA36211s/2022|2022-11-16...,NaN,NaN,{'': 1},NaN,NaN,{},{},{},...,{},{},{},{},0.000000,NaN,NaN,NaN,NaN,0.000000
3,3,hCoV-19/Germany/MV-RKI-I-641916/2022|2022-02-2...,T16494C,T>C,{'T>C': 1},ATG,A[T>C]G,{},{},{},...,{},{},{},{'A[T>C]G': 1.0},-0.528901,NaN,NaN,NaN,-0.356234,-0.885135
4,4,hCoV-19/Germany/BW-RKI-I-354764/2021|2021-11-2...,"C2227A,G4207T,C13673A,A21603G,C21658T,C23655T,...","C>A,G>T,C>A,A>G,C>T,C>T,C>T,G>T,C>T","{'C>A': 2, 'G>T': 2, 'A>G': 1, 'C>T': 4}","CCT,CGA,TCT,CAG,TCA,TCA,ACC,CGA,ACG","C[C>A]T,C[G>T]A,T[C>A]T,C[A>G]G,T[C>T]A,T[C>T]...",{},{'C[A>G]G': 1},"{'T[C>T]A': 2, 'A[C>T]C': 1, 'A[C>T]G': 1}",...,{},{'C[A>G]G': 1.0},"{'T[C>T]A': 0.5, 'A[C>T]C': 0.25, 'A[C>T]G': 0...",{},-4.058362,NaN,0.285154,-0.176959,NaN,-3.950167


In [8]:
df_mov = df[df["sum_llrs"] > 6]
print(len(df_mov))
print(df_mov.head())


1910
       Unnamed: 0                                            seqName  \
1493         1493  hCoV-19/Germany/BY-RKI-I-244904/2021|2021-09-1...   
4274         4274  hCoV-19/Germany/NW-HHU-30660/2022|2022-07-20|2...   
8619         8619  hCoV-19/Slovakia/KE_22_00000844/2022|2022-04-1...   
15896       15896  hCoV-19/USA/NY-PRL-2021_0930_01C11/2021|2021-0...   
24465       24465  hCoV-19/Brazil/MA-FIOCRUZ-6871/2021|2021-01-04...   

              privateNucMutations.unlabeledSubstitutions  \
1493   G1400A,G3443A,C8829T,C9050T,G9658A,C11001T,C20...   
4274   C222T,C512T,G515A,C751T,T1702C,G2144A,A2500G,C...   
8619   G1743A,G2804A,C4686T,G5690A,C6278T,G6479A,C654...   
15896  G581A,T1617C,C2119T,C2910T,C7926T,C14657T,C208...   
24465  C337T,C2449T,C2623T,C4927T,C7819T,C10376T,C104...   

                                                    subs  \
1493         G>A,G>A,C>T,C>T,G>A,C>T,C>T,C>T,C>T,C>T,C>T   
4274   C>T,C>T,G>A,C>T,T>C,G>A,A>G,C>T,G>A,G>T,G>A,T>...   
8619   G>A,G>A,C>T,G>

In [11]:
df_mov.drop(columns=['Unnamed: 0', 'privateNucMutations.unlabeledSubstitutions', 'subs', 'Counts', 'context', 'spectrum', 'G>A_counts', 'A>G_counts', 'G>Aproportions', 'C>Tproportions', 'A>Gproportions', 'T>Cproportions',
                     'C>T_counts', 'T>C_counts','G>A_llr', 'A>G_llr',
       'C>T_llr', 'T>C_llr', 'sum_llrs'], inplace=True)

/var/folders/bt/jpy5j9ms7pb6p6hhqzvpjffw0000gn/T/ipykernel_35022/875515166.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_mov.drop(columns=['Unnamed: 0', 'privateNucMutations.unlabeledSubstitutions', 'subs', 'Counts', 'context', 'spectrum', 'G>A_counts', 'A>G_counts', 'G>Aproportions', 'C>Tproportions', 'A>Gproportions', 'T>Cproportions',


In [12]:
df_mov.to_csv("/Users/reem/mov_2026.tsv", sep="\t", index=False)

In [41]:
df = df.drop(['Unnamed: 0','privateNucMutations.unlabeledSubstitutions','subs','Counts','context','spectrum','G>A_counts','A>G_counts','C>T_counts','T>C_counts','G>Aproportions','A>Gproportions','C>Tproportions','T>Cproportions'],axis=1)
df.head()

,seqName,LLR,G>A_llr,A>G_llr,C>T_llr,T>C_llr
0,hCoV-19/USA/CA-CDPH-500004296/2021|2021-07-17|...,0.133011,NaN,0.004225,0.069622,0.156313
1,hCoV-19/Spain/CL-COV01948/2021|2021-04-08|2021...,-1.049009,NaN,0.319879,0.480353,NaN
2,hCoV-19/USA/OR-OHSU-213401246/2021|2021-10-27|...,1.231103,NaN,-1.811061,-0.505227,-0.447496
3,hCoV-19/Germany/SL-RKI-I-1077947/2022|2022-12-...,-0.210447,NaN,0.004225,NaN,NaN
4,hCoV-19/USA/MA-CDCBI-CRSP_HGQQM7RZS5PYBHGU/202...,-1.073550,NaN,NaN,-0.489601,NaN


In [42]:
df.to_csv("/Users/reem/Mov/llrs_only.tsv",sep= "\t")


In [2]:
import pandas as pd
file_path = "/Users/reem/Mov/nextclade_results/"
for f in ["GtoAprobs_new.tsv", "AtoGprobs_new.tsv", "CtoTprobs_new.tsv", "TtoCprobs_new.tsv"]:
    t = pd.read_csv(file_path + f, sep="\t")  # use your actual filenames
    print(f, "zero cells:", (t[["Molnupiravir", "Normal"]] == 0).sum().sum())

GtoAprobs_new.tsv zero cells: 0
AtoGprobs_new.tsv zero cells: 0
CtoTprobs_new.tsv zero cells: 0
TtoCprobs_new.tsv zero cells: 0
